# $\eta$ decay kinematics check (GENIE PR #514 / Pythia8 stuck-seed bug)

[GENIE-MC/Generator PR #514](https://github.com/GENIE-MC/Generator/pull/514) fixes a bug where Pythia8's random seed
was never actually applied (Pythia8 `Settings::init()` returned early), so every job silently reused the default seed
65539. SBND saw anomalous $\eta\to\gamma\gamma$ photon angular distributions with GENIE v3.06.02 + Pythia8.

This notebook checks whether anything like that is visible in the MCC9.10 BNB $\nu$ overlay samples used by this framework.

**Where the information lives.** `all_df.parquet` only keeps counts (`true_num_prim_gamma`, ...), not the per-particle truth
arrays, so we go back to the checkout ROOT files. The gLEE `singlephotonana/vertex_tree` stores the full GENIE particle record
(`mctruth_daughters_*` with status codes and mother track IDs). The $\eta$ (PDG 221) is decayed **inside GENIE**
(status code 3 = decayed state); it never appears in the Geant4 tree (`wcpselection/T_PFeval`), where its photons show up as
mother-0 primaries. That is exactly what the `eta_other` category in `src/signal_categories.py` relies on
(`true_num_prim_gamma >= 2`, no $\pi^0$).

**What we test**, using the $\eta$'s own 4-momentum from the GENIE record to boost its daughters to the rest frame:

1. decay-mode branching fractions (Pythia6 vs Pythia8 tables differ slightly);
2. isotropy of $\cos\theta^*$ and $\phi^*$ in the $\eta$ rest frame, relative to the $\eta$ lab direction and to the detector axes;
3. the lab-frame consequence, $(E_1-E_2)/(E_1+E_2) = \beta\cos\theta^*$;
4. the stuck-seed signature: exactly repeated rest-frame decay configurations across jobs;
5. a cross-check that the Wire-Cell truth tree photons feeding `eta_other` are these same GENIE photons.

Files: Run 4b, Run 1-3 `hist_1`, Run 5 $\nu$ overlay. Extraction takes a few minutes per file, so results are cached in
`eta_decay_check_cache/`; delete a cache file to re-extract.

In [ ]:
import os, numpy as np, awkward as ak, uproot
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy import stats
from collections import Counter
mpl.rcParams['figure.dpi'] = 110

DATA_DIR = "/nevis/riverside/data/leehagaman/ngem/data_files"
CACHE_DIR = "eta_decay_check_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

FILES = {
    "Run 4b":          "checkout_MCC9.10_Run4b_v10_04_07_20_BNB_nu_overlay_retuple_retuple_hist.root",
    "Run 1-3 hist_1":  "checkout_MCC9.10_Run123_v10_04_07_20_BNB_nu_overlay_surprise_reco2_hist_1.root",
    "Run 5":           "checkout_MCC9.10_Run4acd5_v10_04_07_20_BNB_nu_overlay_retuple_retuple_hist_5.root",
}
CACHES = {"Run 4b": "genie_eta_run4b.npz", "Run 1-3 hist_1": "genie_eta_run123_hist1.npz", "Run 5": "genie_eta_run5.npz"}

M_ETA_GENIE = 0.54745   # GeV, value in the GENIE/ROOT PDG table (and Pythia6); PDG 2024 is 0.547862

## 1. Extract every $\eta$ from the GENIE record (cached)

In [ ]:
def extract_genie_etas(path, step_size=20000):
    """Return dict of per-eta arrays from singlephotonana/vertex_tree (GENIE MCTruth particle record)."""
    t = uproot.open(path)["singlephotonana/vertex_tree"]
    br = ['run_number','subrun_number','event_number','mctruth_daughters_pdg','mctruth_daughters_status_code',
          'mctruth_daughters_trackID','mctruth_daughters_mother_trackID','mctruth_daughters_E',
          'mctruth_daughters_px','mctruth_daughters_py','mctruth_daughters_pz']
    out = {k: [] for k in ['run','subrun','event','eta_E','eta_px','eta_py','eta_pz','eta_status','eta_trackID','ndau',
                           'dau_pdg','dau_status','dau_E','dau_px','dau_py','dau_pz']}
    pi0_status, eta_status_all = [], []
    n_seen = 0
    for arrs in t.iterate(br, step_size=step_size, library='ak'):
        pdg = arrs['mctruth_daughters_pdg']
        n_seen += len(pdg)
        pi0_status.append(np.asarray(ak.flatten(arrs['mctruth_daughters_status_code'][pdg == 111])))
        eta_status_all.append(np.asarray(ak.flatten(arrs['mctruth_daughters_status_code'][pdg == 221])))
        sub = arrs[ak.any(pdg == 221, axis=1)]
        for ev in sub:
            p = np.asarray(ev['mctruth_daughters_pdg']); st = np.asarray(ev['mctruth_daughters_status_code'])
            tid = np.asarray(ev['mctruth_daughters_trackID']); mo = np.asarray(ev['mctruth_daughters_mother_trackID'])
            E = np.asarray(ev['mctruth_daughters_E']); px = np.asarray(ev['mctruth_daughters_px'])
            py = np.asarray(ev['mctruth_daughters_py']); pz = np.asarray(ev['mctruth_daughters_pz'])
            for j in np.where(p == 221)[0]:
                d = mo == tid[j]
                out['run'].append(ev['run_number']); out['subrun'].append(ev['subrun_number']); out['event'].append(ev['event_number'])
                out['eta_E'].append(E[j]); out['eta_px'].append(px[j]); out['eta_py'].append(py[j]); out['eta_pz'].append(pz[j])
                out['eta_status'].append(st[j]); out['eta_trackID'].append(tid[j]); out['ndau'].append(d.sum())
                for k, arr in [('dau_pdg', p), ('dau_status', st), ('dau_E', E), ('dau_px', px), ('dau_py', py), ('dau_pz', pz)]:
                    out[k].append(arr[d])
        print(f"  {n_seen} events scanned, {len(out['eta_E'])} etas", end='\r')
    print()
    res = {k: (np.array(v, dtype=object) if k.startswith('dau_') else np.array(v)) for k, v in out.items()}
    res['pi0_status'] = np.concatenate(pi0_status); res['eta_status_all'] = np.concatenate(eta_status_all)
    return res

data = {}
for label, fname in FILES.items():
    cpath = os.path.join(CACHE_DIR, CACHES[label])
    if not os.path.exists(cpath):
        print("extracting", label)
        np.savez(cpath, **extract_genie_etas(os.path.join(DATA_DIR, fname)))
    data[label] = dict(np.load(cpath, allow_pickle=True))
    print(f"{label:16s}: {len(data[label]['eta_E'])} eta entries")

## 2. Status codes and decay modes

GENIE status codes: 14 = hadron inside the nucleus (pre-FSI), 3 = decayed state, 1 = stable final state, 12 = decayed-inside-nucleus
placeholder. Every $\eta$ shows up twice: once as a status-14 particle whose only "daughter" is the post-FSI status-3 copy, which then
decays. $\pi^0$s by contrast are status 1: GENIE hands them to Geant4 undecayed. So **the $\eta$ decay is done by GENIE's decayer**, and
that is the code path PR #514 is about.

In [ ]:
for label, d in data.items():
    print(label)
    print("   eta status codes:", dict(zip(*np.unique(d['eta_status_all'], return_counts=True))))
    print("   pi0 status codes:", dict(zip(*np.unique(d['pi0_status'], return_counts=True))))

def cat(key): return np.concatenate([d[key] for d in data.values()])
src = np.concatenate([[label]*len(d['eta_E']) for label, d in data.items()])
dau_pdg = cat('dau_pdg')
eta_p4 = np.column_stack([cat('eta_E'), cat('eta_px'), cat('eta_py'), cat('eta_pz')]).astype(float)
dau_E, dau_px, dau_py, dau_pz = cat('dau_E'), cat('dau_px'), cat('dau_py'), cat('dau_pz')
run, subrun, event = cat('run'), cat('subrun'), cat('event')

mode_key = np.array([tuple(sorted(int(x) for x in p)) for p in dau_pdg], dtype=object)
decayed = np.array([k != (221,) and len(k) > 0 for k in mode_key])
def is_mode(m): return np.array([k == m for k in mode_key], dtype=bool)
print(f"\nTotal decayed etas over the three files: {decayed.sum()}")

In [ ]:
# branching fractions vs PDG, Pythia6 and Pythia8 decay tables
MODES = [((22,22), r"$\gamma\gamma$"), ((111,111,111), r"$3\pi^0$"), ((-211,111,211), r"$\pi^+\pi^-\pi^0$"),
         ((-211,22,211), r"$\pi^+\pi^-\gamma$"), ((-11,11,22), r"$e^+e^-\gamma$"), ((-13,13,22), r"$\mu^+\mu^-\gamma$")]
PDG_BR     = [0.3936, 0.3257, 0.2302, 0.0428, 0.0069, 0.00031]      # PDG 2024
PYTHIA6_BR = [0.3925, 0.3200, 0.2260, 0.0468, 0.0050, 0.00031]      # Pythia 6.4 decay table (MDME entries for eta)
PYTHIA8_BR = [0.3931, 0.3257, 0.2274, 0.0422, 0.0069, 0.00031]      # Pythia 8 ParticleData.xml

ndec = decayed.sum()
frac = np.array([np.sum(is_mode(m) & decayed) for m, _ in MODES]) / ndec
err = np.sqrt(frac*(1-frac)/ndec)
counts = Counter(mode_key[decayed].tolist())
print("observed modes:")
for k, v in counts.most_common(10): print(f"   {k}: {v}  ({100*v/ndec:.2f}%)")

fig, ax = plt.subplots(figsize=(8,4))
x = np.arange(len(MODES)); w = 0.2
ax.bar(x-1.5*w, PDG_BR, w, label='PDG 2024'); ax.bar(x-0.5*w, PYTHIA6_BR, w, label='Pythia 6.4 table')
ax.bar(x+0.5*w, PYTHIA8_BR, w, label='Pythia 8 table')
ax.errorbar(x+1.5*w, frac, yerr=err, fmt='ko', label=f'this sample (N={ndec})')
ax.set_xticks(x); ax.set_xticklabels([l for _, l in MODES]); ax.set_yscale('log'); ax.set_ylim(1e-4, 1)
ax.set_ylabel('branching fraction'); ax.legend(); ax.set_title(r'$\eta$ decay modes in the GENIE record')
plt.show()
for (m, l), f_, e_ in zip(MODES, frac, err):
    print(f"{l:22s} obs {100*f_:6.2f} +- {100*e_:.2f} %   P6 {100*PYTHIA6_BR[MODES.index((m,l))]:.2f}   P8 {100*PYTHIA8_BR[MODES.index((m,l))]:.2f}")

The $\pi^+\pi^-\gamma$ and $e^+e^-\gamma$ fractions sit on the Pythia6 table rather than Pythia8's, and the $\eta$ mass in the
record (below) is the Pythia6/ROOT value 0.54745 GeV. Together with the v3.0.6 `UBGenie` knob set in `spline_weights`, that says
this production is Pythia6-era GENIE, i.e. the Pythia8 seeding path of PR #514 was never exercised. The kinematic tests below check
the decays directly anyway.

## 3. $\eta\to\gamma\gamma$: rest-frame kinematics

In [ ]:
def boost(p4, beta):
    """Boost 4-vectors p4 (N,4)=(E,px,py,pz) into the frame moving with velocity beta (N,3)."""
    b2 = np.sum(beta**2, axis=1); g = 1/np.sqrt(1-b2); bp = np.sum(beta*p4[:,1:], axis=1)
    E = g*(p4[:,0] - bp)
    p = p4[:,1:] + ((g-1)/b2*bp - g*p4[:,0])[:,None]*beta
    return np.column_stack([E, p])

gg = np.where(is_mode((22,22)))[0]
g1 = np.array([[dau_E[i][0], dau_px[i][0], dau_py[i][0], dau_pz[i][0]] for i in gg], dtype=float)
g2 = np.array([[dau_E[i][1], dau_px[i][1], dau_py[i][1], dau_pz[i][1]] for i in gg], dtype=float)
eta = eta_p4[gg]
p_eta = np.linalg.norm(eta[:,1:], axis=1); beta_eta = p_eta/eta[:,0]
eta_dir = eta[:,1:]/p_eta[:,None]
m_eta_rec = np.sqrt(eta[:,0]**2 - p_eta**2)
s = g1 + g2
m_gg = np.sqrt(np.maximum(s[:,0]**2 - np.sum(s[:,1:]**2, axis=1), 0))

g1r = boost(g1, eta[:,1:]/eta[:,0][:,None]); g2r = boost(g2, eta[:,1:]/eta[:,0][:,None])
u1 = g1r[:,1:]/np.linalg.norm(g1r[:,1:], axis=1)[:,None]          # rest-frame direction of the first-listed photon
cos_star = np.sum(u1*eta_dir, axis=1)                             # w.r.t. the eta lab direction
# azimuth around the eta direction
ref = np.cross(eta_dir, [0,1,0.]); bad = np.linalg.norm(ref,axis=1) < 1e-6; ref[bad] = np.cross(eta_dir[bad], [1,0,0.])
ref /= np.linalg.norm(ref, axis=1)[:,None]; ref2 = np.cross(eta_dir, ref)
phi_star = np.arctan2(np.sum(u1*ref2, axis=1), np.sum(u1*ref, axis=1))

print(f"N(eta->gg) = {len(gg)}")
print(f"eta mass from record: {m_eta_rec.mean():.5f} +- {m_eta_rec.std():.1e} GeV;  m(gg): {m_gg.mean():.5f} +- {m_gg.std():.1e} GeV")
print(f"max |p4(g1)+p4(g2) - p4(eta)| = {np.abs(s-eta).max():.1e} GeV")
print(f"rest-frame photon energies: {g1r[:,0].mean():.5f}, {g2r[:,0].mean():.5f} (expect m/2 = {M_ETA_GENIE/2:.5f});  |p1*+p2*| max = {np.linalg.norm(g1r[:,1:]+g2r[:,1:],axis=1).max():.1e}")

fig, axs = plt.subplots(1, 3, figsize=(14,3.6))
axs[0].hist(p_eta, bins=50, range=(0,3)); axs[0].set_xlabel(r'$\eta$ lab momentum [GeV]'); axs[0].set_ylabel('etas')
axs[1].hist(beta_eta, bins=50, range=(0,1)); axs[1].set_xlabel(r'$\beta_\eta$')
axs[2].hist((m_gg-M_ETA_GENIE)*1e6, bins=50); axs[2].set_xlabel(r'$m_{\gamma\gamma} - 0.54745$ GeV  [eV]'); axs[2].set_title('float32 round-off only')
plt.tight_layout(); plt.show()

In [ ]:
def flat_hist(ax, x, lo, hi, nb, xlabel, title=None, weights=None):
    """Histogram with the flat expectation and chi2/KS p-values vs uniform."""
    h, edges = np.histogram(x, bins=nb, range=(lo,hi))
    exp = len(x)/nb
    chi2 = np.sum((h-exp)**2/exp); p_chi2 = stats.chi2.sf(chi2, nb-1)
    p_ks = stats.kstest(x, 'uniform', args=(lo, hi-lo)).pvalue
    c = 0.5*(edges[1:]+edges[:-1])
    ax.errorbar(c, h/exp, yerr=np.sqrt(h)/exp, fmt='o', ms=4)
    ax.axhline(1, color='k', lw=1, ls='--')
    ax.set_xlabel(xlabel); ax.set_ylabel('ratio to flat')
    ax.set_ylim(0.6, 1.4)
    ax.set_title((title or '') + f"  N={len(x)}\n$\\chi^2$/ndf={chi2:.1f}/{nb-1} (p={p_chi2:.2f}), KS p={p_ks:.2f}", fontsize=9)
    return chi2, p_chi2, p_ks

fig, axs = plt.subplots(1, 3, figsize=(14,3.8))
flat_hist(axs[0], cos_star, -1, 1, 20, r'$\cos\theta^*$ (photon 1 in $\eta$ rest frame, w.r.t. $\eta$ lab direction)')
flat_hist(axs[1], np.abs(cos_star), 0, 1, 10, r'$|\cos\theta^*|$ w.r.t. $\eta$ lab direction')
flat_hist(axs[2], phi_star, -np.pi, np.pi, 12, r'$\phi^*$ around $\eta$ lab direction')
plt.suptitle(r'Isotropy of $\eta\to\gamma\gamma$ in the $\eta$ rest frame (all three files)', y=1.03); plt.tight_layout(); plt.show()

fig, axs = plt.subplots(1, 3, figsize=(14,3.8))
for a, nm in enumerate('xyz'):
    flat_hist(axs[a], u1[:,a], -1, 1, 10, rf'rest-frame $\cos\theta^*$ w.r.t. detector ${nm}$ axis')
plt.suptitle('Rest-frame direction relative to fixed detector axes (a stuck seed would pile up at the same directions)', y=1.03)
plt.tight_layout(); plt.show()

In [ ]:
# per-file breakdown of the main observable
fig, axs = plt.subplots(1, 3, figsize=(14,3.8))
for ax, label in zip(axs, data):
    m = src[gg] == label
    flat_hist(ax, cos_star[m], -1, 1, 10, r'$\cos\theta^*$ w.r.t. $\eta$ direction', title=label)
plt.tight_layout(); plt.show()

# and vs eta momentum: a bad boost / bad rest-frame sampling typically shows up as a p-dependent asymmetry
fig, axs = plt.subplots(1, 2, figsize=(11,3.8))
axs[0].hist2d(p_eta, cos_star, bins=[np.linspace(0,2,21), np.linspace(-1,1,11)], cmap='viridis')
axs[0].set_xlabel(r'$p_\eta$ [GeV]'); axs[0].set_ylabel(r'$\cos\theta^*$')
bins = [0, 0.3, 0.5, 0.7, 1.0, 3.0]
means = [cos_star[(p_eta>=lo)&(p_eta<hi)].mean() for lo, hi in zip(bins[:-1], bins[1:])]
errs  = [cos_star[(p_eta>=lo)&(p_eta<hi)].std()/np.sqrt(((p_eta>=lo)&(p_eta<hi)).sum()) for lo, hi in zip(bins[:-1], bins[1:])]
axs[1].errorbar(0.5*(np.array(bins[:-1])+np.array(bins[1:])), means, yerr=errs, fmt='o'); axs[1].axhline(0, color='k', ls='--', lw=1)
axs[1].set_xlabel(r'$p_\eta$ [GeV]'); axs[1].set_ylabel(r'$\langle\cos\theta^*\rangle$ (expect 0)')
print(f"corr(cos*, beta) = {np.corrcoef(cos_star, beta_eta)[0,1]:+.3f}, corr(cos*, E_eta) = {np.corrcoef(cos_star, eta[:,0])[0,1]:+.3f}")
plt.tight_layout(); plt.show()

### Lab-frame view
For a two-body decay of a particle with velocity $\beta$, $E_{1,2} = \gamma\frac{m}{2}(1\pm\beta\cos\theta^*)$, so the lab
energy asymmetry $(E_1-E_2)/(E_1+E_2)$ must equal $\beta\cos\theta^*$ and be flat in $[-\beta,\beta]$. The minimum lab opening angle is
$2\arcsin(1/\gamma)$. These are the quantities that would be visibly wrong in the reconstructed photons if the rest-frame sampling were off.

In [ ]:
asym = (g1[:,0]-g2[:,0])/(g1[:,0]+g2[:,0])
cos_open = np.sum(g1[:,1:]*g2[:,1:], axis=1)/(np.linalg.norm(g1[:,1:],axis=1)*np.linalg.norm(g2[:,1:],axis=1))
open_deg = np.degrees(np.arccos(np.clip(cos_open, -1, 1)))
gamma_eta = eta[:,0]/m_eta_rec
min_open = np.degrees(2*np.arcsin(np.clip(1/gamma_eta, 0, 1)))

fig, axs = plt.subplots(1, 3, figsize=(14,3.8))
axs[0].scatter(beta_eta*cos_star, asym, s=3, alpha=0.4); axs[0].plot([-1,1],[-1,1],'k--',lw=1)
axs[0].set_xlabel(r'$\beta\cos\theta^*$'); axs[0].set_ylabel(r'$(E_1-E_2)/(E_1+E_2)$'); axs[0].set_title(f'max deviation {np.abs(asym-beta_eta*cos_star).max():.1e}', fontsize=9)
flat_hist(axs[1], np.clip(asym/beta_eta, -1, 1), -1, 1, 10, r'$(E_1-E_2)/(E_1+E_2)\,/\,\beta$')
axs[2].scatter(p_eta, open_deg, s=3, alpha=0.4, label='decays'); o = np.argsort(p_eta); axs[2].plot(p_eta[o], min_open[o], 'r-', lw=1, label=r'$2\arcsin(1/\gamma)$')
axs[2].set_xlabel(r'$p_\eta$ [GeV]'); axs[2].set_ylabel('lab opening angle [deg]'); axs[2].legend(); axs[2].set_xlim(0, 2.5)
plt.tight_layout(); plt.show()
print(f"events with opening angle below the kinematic minimum: {(open_deg < min_open - 1e-3).sum()}")

## 4. Stuck-seed signature: repeated random sequences across jobs

If every job used the same Pythia seed, the $k$-th Pythia decay in every job would draw the *same* random numbers, and identical
rest-frame configurations would recur across the sample. We count exact repeats of the rest-frame photon direction
($\gamma\gamma$) and of the sorted rest-frame daughter energies (three-body modes), and look at the autocorrelation of
$\cos\theta^*$ in run/subrun/event order.

In [ ]:
def count_dupes(keys):
    c = Counter(keys); return sum(v for v in c.values() if v > 1), sum(1 for v in c.values() if v > 1)

n_dup, n_grp = count_dupes([tuple(np.round(v, 5)) for v in u1])
print(f"eta->gg: {n_dup} decays in {n_grp} groups share an identical rest-frame direction (5 decimals)")
# show them
groups = {}
for k, v in enumerate(u1): groups.setdefault(tuple(np.round(v, 5)), []).append(k)
for key, ks in groups.items():
    if len(ks) > 1:
        print("  identical rest-frame direction:")
        for k in ks:
            i = gg[k]; print(f"     {src[i]:16s} run/subrun/event {run[i]}/{subrun[i]}/{event[i]}   eta p4 = {np.round(eta[k],5)}")

for mode, name in [((111,111,111), '3pi0'), ((-211,111,211), 'pi+pi-pi0'), ((-211,22,211), 'pi+pi-gamma')]:
    sel = np.where(is_mode(mode))[0]
    trip = []
    for i in sel:
        p4 = np.column_stack([dau_E[i], dau_px[i], dau_py[i], dau_pz[i]]).astype(float)
        b = (eta_p4[i,1:]/eta_p4[i,0])[None,:].repeat(len(p4), 0)
        trip.append(tuple(np.round(np.sort(boost(p4, b)[:,0]), 5)))
    n_dup, n_grp = count_dupes(trip)
    print(f"{name:12s}: {n_dup} / {len(sel)} decays share identical rest-frame energies")

order = np.lexsort((event[gg], subrun[gg], run[gg])); cs = cos_star[order]
lags = np.arange(1, 11)
ac = [np.corrcoef(cs[:-l], cs[l:])[0,1] for l in lags]
fig, ax = plt.subplots(figsize=(6,3.2))
ax.bar(lags, ac); ax.axhspan(-1/np.sqrt(len(cs)), 1/np.sqrt(len(cs)), color='gray', alpha=0.3, label=r'$\pm1/\sqrt{N}$')
ax.set_xlabel('lag (etas, in run/subrun/event order)'); ax.set_ylabel(r'autocorrelation of $\cos\theta^*$'); ax.legend(); plt.show()

The only two repeated $\gamma\gamma$ configurations have **identical $\eta$ 4-momenta as well**, i.e. they are whole duplicated
GENIE events (runs 25097 and 25117 in the Run 5 file), not a decay-sampling coincidence. That is a separate production quirk; see
`duplicated_genie_events.ipynb`.

## 5. Three-body modes as a further check of the decayer
For $\eta\to3\pi^0$ the Dalitz plot is (almost) uniform; for $\eta\to\pi^+\pi^-\pi^0$ the matrix element gives a mild linear slope
in the $\pi^0$ energy. Either way the three rest-frame energies should have identical marginal distributions.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(14,3.8))
for ax, (mode, name) in zip(axs[:2], [((111,111,111), r'$\eta\to3\pi^0$'), ((-211,111,211), r'$\eta\to\pi^+\pi^-\pi^0$')]):
    sel = np.where(is_mode(mode))[0]
    Er = []
    for i in sel:
        p4 = np.column_stack([dau_E[i], dau_px[i], dau_py[i], dau_pz[i]]).astype(float)
        b = (eta_p4[i,1:]/eta_p4[i,0])[None,:].repeat(len(p4), 0)
        Er.append(boost(p4, b)[:,0])
    Er = np.array(Er)
    for slot in range(3):
        ax.hist(Er[:,slot], bins=30, range=(0.13, 0.30), histtype='step', label=f'daughter slot {slot} (mean {Er[:,slot].mean():.4f})')
    ax.set_xlabel('rest-frame daughter energy [GeV]'); ax.set_title(f'{name}  N={len(sel)}', fontsize=10); ax.legend(fontsize=7)
    if mode == (-211,111,211):
        # Dalitz variables: X = sqrt3 (T+ - T-)/Q, Y = 3 T0/Q - 1
        pdgs = np.array([np.asarray(dau_pdg[i]) for i in sel])
        T = Er - np.array([[0.13957 if abs(p)==211 else 0.13498 for p in row] for row in pdgs])
        Q = T.sum(1)
        Tp = np.array([t[p==211][0] for t,p in zip(T,pdgs)]); Tm = np.array([t[p==-211][0] for t,p in zip(T,pdgs)]); T0 = np.array([t[p==111][0] for t,p in zip(T,pdgs)])
        X = np.sqrt(3)*(Tp-Tm)/Q; Y = 3*T0/Q - 1
        axs[2].hist2d(X, Y, bins=[np.linspace(-1.1,1.1,23), np.linspace(-1.1,1.1,23)], cmap='viridis')
        axs[2].set_xlabel(r'$X=\sqrt{3}(T_+-T_-)/Q$'); axs[2].set_ylabel(r'$Y=3T_0/Q-1$'); axs[2].set_title(r'$\pi^+\pi^-\pi^0$ Dalitz plot', fontsize=10)
plt.tight_layout(); plt.show()

## 6. Cross-check against the Wire-Cell truth tree (what `eta_other` actually uses)

`postprocessing.add_extra_true_photon_variables` counts photons with `truth_mother == 0` in `wcpselection/T_PFeval`
(Geant4 tree). Here we pull the events with $\ge2$ primary photons from the Run 4b file, match them by run/subrun/event to the GENIE
$\eta\to\gamma\gamma$ list, and repeat the rest-frame test using the summed photon 4-momentum as the $\eta$ (which is all the framework
would have).

In [ ]:
def extract_wc_primary_gammas(path, step_size=20000):
    t = uproot.open(path)["wcpselection/T_PFeval"]
    out = {k: [] for k in ['run','subrun','event','n_prim_gamma','n_pi0','g_E','g_px','g_py','g_pz','pdg_prim']}
    n_eta_pdg = 0
    for arrs in t.iterate(['run','subrun','event','truth_pdg','truth_mother','truth_startMomentum'], step_size=step_size, library='ak'):
        pdg, mo = arrs['truth_pdg'], arrs['truth_mother']
        n_eta_pdg += int(ak.sum(pdg == 221))
        prim_g = (pdg == 22) & (mo == 0)
        sel = ak.sum(prim_g, axis=1) >= 2
        sub, pg = arrs[sel], prim_g[sel]
        for i in range(len(sub)):
            m = np.asarray(sub[i]['truth_startMomentum'][pg[i]])
            out['run'].append(sub[i]['run']); out['subrun'].append(sub[i]['subrun']); out['event'].append(sub[i]['event'])
            out['n_prim_gamma'].append(len(m)); out['n_pi0'].append(int(ak.sum(sub[i]['truth_pdg'] == 111)))
            out['g_px'].append(m[:,0]); out['g_py'].append(m[:,1]); out['g_pz'].append(m[:,2]); out['g_E'].append(m[:,3])
            out['pdg_prim'].append(np.asarray(sub[i]['truth_pdg'][sub[i]['truth_mother'] == 0]))
    res = {k: (np.array(v, dtype=object) if k.startswith('g_') or k == 'pdg_prim' else np.array(v)) for k, v in out.items()}
    res['n_eta_pdg'] = n_eta_pdg
    return res

wc_cache = os.path.join(CACHE_DIR, "wc_primgamma_run4b.npz")
if not os.path.exists(wc_cache):
    np.savez(wc_cache, **extract_wc_primary_gammas(os.path.join(DATA_DIR, FILES["Run 4b"])))
w = dict(np.load(wc_cache, allow_pickle=True))
print("PDG 221 entries in the Geant4 tree (T_PFeval):", int(w['n_eta_pdg']), " -> the eta is decayed before Geant4")
print("events with >=2 primary photons:", len(w['run']), "; n_prim_gamma:", dict(zip(*np.unique(w['n_prim_gamma'], return_counts=True))))

d4b = data["Run 4b"]
gg4b = np.array([tuple(sorted(int(x) for x in p)) == (22,22) for p in d4b['dau_pdg']])
rse_genie = {(int(r),int(s),int(e)): sorted(d4b['dau_E'][i].tolist()) for i, (r,s,e) in enumerate(zip(d4b['run'], d4b['subrun'], d4b['event'])) if gg4b[i]}
rse_wc = list(zip(w['run'].tolist(), w['subrun'].tolist(), w['event'].tolist()))
in_genie = np.array([r in rse_genie for r in rse_wc])
sel2 = (w['n_prim_gamma'] == 2) & (w['n_pi0'] == 0)      # the eta_other-like topology
print(f"GENIE eta->gg events: {len(rse_genie)}; all found in the WC >=2-primary-photon list: {in_genie.sum()}")
print(f"WC exactly-2-primary-photon, 0-pi0 events: {sel2.sum()}, of which GENIE eta->gg: {(sel2 & in_genie).sum()}  ({100*(sel2&in_genie).sum()/sel2.sum():.1f}%)")

idx = np.where(sel2)[0]
E  = np.array([w['g_E'][i] for i in idx]); px = np.array([w['g_px'][i] for i in idx]); py = np.array([w['g_py'][i] for i in idx]); pz = np.array([w['g_pz'][i] for i in idx])
P4 = np.stack([E.sum(1), px.sum(1), py.sum(1), pz.sum(1)], 1)
m_wc = np.sqrt(np.maximum(P4[:,0]**2 - np.sum(P4[:,1:]**2, 1), 0))
isG = in_genie[idx]
dE = [max(abs(a-b) for a, b in zip(sorted(E[k].tolist()), rse_genie[rse_wc[idx[k]]])) for k in range(len(idx)) if isG[k]]
print(f"max |E_gamma(WC) - E_gamma(GENIE)| over matched events: {max(dE):.1e} GeV")

wdir = P4[:,1:]/np.linalg.norm(P4[:,1:], axis=1)[:,None]
p1 = np.stack([px[:,0], py[:,0], pz[:,0]], 1)
p1r = boost(np.column_stack([E[:,0], p1]), P4[:,1:]/P4[:,0][:,None])[:,1:]
cos_wc = np.sum(p1r*wdir, 1)/np.linalg.norm(p1r, axis=1)

fig, axs = plt.subplots(1, 2, figsize=(11,3.8))
axs[0].hist(m_wc[isG], bins=60, range=(0, 0.7), label=r'GENIE $\eta\to\gamma\gamma$'); axs[0].hist(m_wc[~isG], bins=60, range=(0, 0.7), label='other 2-primary-photon events')
axs[0].set_xlabel(r'$m_{\gamma\gamma}$ of the two primary photons in T_PFeval [GeV]'); axs[0].set_yscale('log'); axs[0].legend(); axs[0].set_title('Run 4b, exactly 2 primary photons, no $\\pi^0$', fontsize=10)
flat_hist(axs[1], cos_wc[isG], -1, 1, 10, r'$\cos\theta^*$ from T_PFeval photons (Run 4b)')
plt.tight_layout(); plt.show()

## Conclusion

* The $\eta$ is decayed by GENIE's decayer, and the sample is Pythia6-era GENIE (v3.0.6 `UBGenie` knobs, Pythia6 branching
  fractions, Pythia6/ROOT $\eta$ mass). The Pythia8-only seeding bug of PR #514 is not in play for this production.
* Directly, the $\eta\to\gamma\gamma$ decays are isotropic in the $\eta$ rest frame: $\cos\theta^*$ and $\phi^*$ are flat relative to
  the $\eta$ direction and to the detector axes, with no dependence on the $\eta$ momentum; the lab energy asymmetry equals
  $\beta\cos\theta^*$ exactly; there are no repeated random configurations across jobs in any decay mode.
* The photons the framework counts for `eta_other` are these same GENIE photons (energies agree to float precision), and ~99 % of the
  exactly-two-primary-photon, no-$\pi^0$ topology is genuine $\eta\to\gamma\gamma$.